# Laplacian Smoothing and Mesh Dual

This notebook contains an exercise where you 
- smooth a mesh by replacing each vertex with the average of its neighbors. 
- compute the volume of a tetrahedral mesh
- compute the dual of a mesh by converting each vertex to a face such that the vertices of the new face correspond to the faces of the old mesh.

In [ ]:
from pygel3d import hmesh, jupyter_display as jd
import numpy as np

jd.set_export_mode(True)
def mesh_stats(m):
    print("# faces : ", m.no_allocated_faces(), " # vertices : ", m.no_allocated_vertices())

### Problem 1: Insert code to complete the smoothing function 

In [ ]:
"""
Smooths the mesh by moving each vertex to the mean position
of its neighboring vertices.
"""
def smooth(m, max_iter=1):
    pos = m.positions()
    new_pos = np.zeros(pos.shape)
    for i in range(0, max_iter):
        for curr_idx in m.vertices():
            neigbors = list(m.circulate_vertex(curr_idx, mode='v'))
            new_pos[curr_idx] = np.mean(pos[neigbors], axis=0)
        pos[:] = new_pos

### Problem 2: Insert code to compute the volume of a triangle mesh

In [ ]:
"""
Calculates the signed volume of a closed triangular mesh
by summing the volumes of tetrahedra formed by each face
and the origin (0, 0, 0).
"""
def calculate_volume(m):
    pos = m.positions()
    vol = 0
    for idx in m.faces():
        vert_idxs = np.array(list(m.circulate_face(idx, mode="v")))        
        p0, p1, p2 = pos[vert_idxs[0]], pos[vert_idxs[1]], pos[vert_idxs[2]]
        vol += np.dot(p0, np.cross(p1, p2)) / 6.
    return vol          

### Problem 3: Insert code to complete the dual function

In [ ]:
"""
Forms a new mesh by switching vertices with faces
"""
def dual(m):
    m2 = hmesh.Manifold()
    pos = m.positions()
    centroids = {}

    for face_idx in m.faces():
        vertex_idxs = list(m.circulate_face(face_idx, mode="v"))
        centroids[face_idx] = np.mean(pos[vertex_idxs], axis=0)

    for vertex_idx in m.vertices():
        face_idxs = list(m.circulate_vertex(vertex_idx, mode="f")) 
        points = []
        for face_idx in face_idxs:
            point = centroids[face_idx]
            points.append(point)

        m2.add_face(points)

    # removes dublicate vertices
    hmesh.stitch(m2)
    return m2

### Show original mesh
If you don't see anything it is probably a problem with the path to `data/bunnygtest.obj`. The path should be relative to the current working directory: probably this is the directory that Jupyter Notebook shows when it starts.

In [ ]:
m = hmesh.load("bunnygtest.obj")
mesh_stats(m)
print("Volume: ", calculate_volume(m))
jd.display(m)

### Show the smoothed mesh

In [ ]:
m_s = hmesh.Manifold(m)
smooth(m_s,50)
print("Volume: ", calculate_volume(m_s))
jd.display(m_s)

### Show the dual mesh

In [ ]:
mesh_stats(m)
m_d = dual(m)
mesh_stats(m_d)
jd.display(m_d)

When everything works, export this to an HTML file and submit that!